In [ ]:
import asyncio
import json
import logging
import os
from typing import List, Dict, Any, Union

from dotenv import load_dotenv

from langchain.messages import HumanMessage
from langchain.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI


from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette import status

load_dotenv()

print("✅ Импорты загружены")

✅ Импорты загружены


In [10]:
import httpx
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

# Устанавливаем прокси вручную (замени на свой)
proxy = "socks5://127.0.0.1:12334"  # Или твой прокси адрес
print(f"🌐 Используется прокси: {proxy}")

# Создаем httpx клиент с прокси
http_client = httpx.Client(proxy=proxy, timeout=30.0)

# Groq клиент с прокси
client = Groq(api_key=groq_api_key, http_client=http_client)

try:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Ты — помощник."},
            {"role": "user", "content": "Привет!"},
        ],
        temperature=0.3,
    )

    print("✅ Ответ через Groq API:")
    print(response.choices[0].message.content)

except Exception as e:
    print(f"❌ Ошибка: {e}")

🌐 Используется прокси: socks5://127.0.0.1:12334
✅ Ответ через Groq API:
Привет! Как я могу вам помочь сегодня?


In [ ]:
import os
import httpx
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

# Устанавливаем прокси вручную (замени на свой)
proxy = "socks5://127.0.0.1:12334"  # Или твой прокси адрес
print(f"🌐 Используется прокси: {proxy}")

# Создаем httpx клиент с прокси
http_client = httpx.Client(proxy=proxy, timeout=30.0)

# Groq клиент с прокси
client = Groq(api_key=groq_api_key, http_client=http_client)

try:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Ты — помощник."},
            {"role": "user", "content": "Привет!"},
        ],
        temperature=0.3,
    )

    print("✅ Ответ через Groq API:")
    print(response.choices[0].message.content)

except Exception as e:
    print(f"❌ Ошибка: {e}")

🌐 Используется прокси: socks5://127.0.0.1:12334
✅ Ответ через Groq API:
Привет! Как я могу вам помочь сегодня?


In [ ]:
import os

# Используем HTTP прокси (который конвертирует в SOCKS5)
http_proxy = "http://127.0.0.1:8888"

os.environ["HTTP_PROXY"] = http_proxy
os.environ["HTTPS_PROXY"] = http_proxy
os.environ["http_proxy"] = http_proxy
os.environ["https_proxy"] = http_proxy

print(f"🌐 Используется HTTP прокси (конвертирует SOCKS5): {http_proxy}")

gemini_api_key = os.getenv("GEMINI_API_KEY")

# Инициализация Gemini
llm = ChatGoogleGenerativeAI(
    google_api_key=gemini_api_key,
    model="gemini-2.5-flash",
    temperature=0.3,
    max_retries=1,
    request_timeout=60,
)

try:
    print("🔄 Отправка запроса через HTTP→SOCKS5 прокси...")
    res = llm.invoke([{"role": "user", "content": "Test"}])
    print("✅ TEST RESULT:", res.content)
except Exception as e:
    print(f"❌ ERROR: {e}")

🌐 Используется HTTP прокси (конвертирует SOCKS5): http://127.0.0.1:8888
🔄 Отправка запроса через HTTP→SOCKS5 прокси...
✅ TEST RESULT: Hello! I'm ready.

How can I help you, or what would you like to test?


### Запуск прокси для Gemini
```bash
python notebooks/http_socks_proxy.py
```

In [ ]:
# --- 1. Инициализация и Конфигурация ---

# ✅ ИСПРАВЛЕНО: Убран неверный 'name', использовано __name__.
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

app = FastAPI()

# ✅ УЛУЧШЕНИЕ: API ключ берется из окружения. Используем gpt-4o-mini для скорости.
llm = ChatGroq(model="gpt-4o-mini", temperature=0.1, api_key=os.getenv("GROQ_API_KEY"))

# ❌ БЫЛО: Небезопасные глобальные переменные.
# ✅ ИСПРАВЛЕНО: Внедрен asyncio.Semaphore для безопасного контроля конкурентности.
MAX_CONCURRENT_REQUESTS = 100
CONCURRENCY_SEMAPHORE = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)


# --- 2. Модели Pydantic ---
class TranslationRequest(BaseModel):
    # ✅ УЛУЧШЕНИЕ: Добавлены Field с описанием.
    text: Union[str, List[str], Dict[str, Any]] = Field(
        ...,
        description="Текст, массив или JSON-объект для перевода",
        max_length=500,  # Добавлена максимальная длина контекста
    )
    source_lang: str = Field("auto", description="Исходный язык.")
    target_lang: str = Field("en", description="Целевой язык.")


class TranslationResponse(BaseModel):
    translated: Union[str, List[str], Dict]
    original: Union[str, List[str], Dict]
    status: str = "success"


# --- 3. Вспомогательные Функции ---


def split_text_into_chunks(text: str, chunk_size: int = 4000) -> List[str]:
    # ❌ БЫЛО: chunk_size = 100 (символов).
    # ✅ ИСПРАВЛЕНО: Увеличен до 4000 (токенов) для лучшего контекста.
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i : i + chunk_size])
    return chunks


# ✅ ИСПРАВЛЕНО: Функция теперь рейзит стандартное исключение (Exception) при ошибке LLM.
async def translate_string(text: str, source_lang: str, target_lang: str) -> str:
    """Переводит длинную строку с разбиением на чанки. При ошибке LLM рейзит Exception."""

    # ❌ БЫЛО: Небезопасное увеличение/уменьшение глобальной active_requests.
    # ✅ ИСПРАВЛЕНО: Контроль конкурентности перенесен в Semaphore в endpoint'е.

    chunks = split_text_into_chunks(text)
    tasks = []

    prompt_template = PromptTemplate(
        input_variables=["text", "source", "target"],
        template="Translate this text from {source} to {target} only, return only the translated text: {text}",
    )

    for chunk in chunks:
        prompt_str = prompt_template.format(
            text=chunk, source=source_lang, target=target_lang
        )
        # ❌ БЫЛО: llm.apredict (устаревший метод).
        # ✅ ИСПРАВЛЕНО: Используем llm.ainvoke (асинхронный вызов).
        tasks.append(llm.ainvoke([HumanMessage(content=prompt_str)]))

    results = await asyncio.gather(*tasks, return_exceptions=True)

    translated_chunks = []

    for r in results:
        if isinstance(r, Exception):
            logger.error(f"Translation chunk failed: {r}")
            # ✅ ИСПРАВЛЕНО: Рейзим стандартное исключение. Endpoint смаппит его в 500.
            raise Exception("Ошибка внешнего API при переводе части текста.")

        translated_chunks.append(r.content)

    return "".join(translated_chunks)


# ✅ НОВАЯ ФУНКЦИЯ: Заменяет неэффективные translate_array и translate_json_recursive.
# ✅ ИСПРАВЛЕНО: Функция теперь рейзит исключения при ошибках (JSONDecodeError или Exception).
async def translate_structured_batch(
    data: Union[List[str], Dict], source_lang: str, target_lang: str
) -> Union[List[str], Dict]:
    """Использует пакетную обработку JSON для перевода массивов и словарей. При ошибке рейзит исключение."""

    # ❌ БЫЛО: Блокирующие вызовы и множественные запросы.
    # ✅ ИСПРАВЛЕНО: Один асинхронный вызов, сохраняющий структуру.

    request_data = {"items": data} if isinstance(data, list) else data

    json_prompt = (
        f"Translate all string values in the following JSON object from {source_lang} to {target_lang}. "
        f"Return ONLY the translated JSON structure, preserve keys and types:\n"
        f"{json.dumps(request_data, ensure_ascii=False)}"
    )

    try:
        raw_translation = await llm.ainvoke([HumanMessage(content=json_prompt)])
        translated_text = raw_translation.content

        translated_json = json.loads(translated_text)

        # Если вход был List, возвращаем только список
        return (
            translated_json.get("items", [])
            if isinstance(data, list)
            else translated_json
        )

    except json.JSONDecodeError as e:
        logger.error(f"LLM returned invalid JSON: {translated_text}. Error: {e}")
        # ✅ ИСПРАВЛЕНО: Рейзим ошибку парсинга, endpoint смаппит ее в 500.
        raise Exception(f"LLM вернул невалидную JSON структуру: {e}")
    except Exception as e:
        logger.error(f"Batch translation failed: {e}")
        # ✅ ИСПРАВЛЕНО: Рейзим общую ошибку API, endpoint смаппит ее в 500.
        raise Exception("Внешняя ошибка API во время пакетного перевода.")


# --- 4. Основной Endpoint ---


# ✅ ИСПРАВЛЕНО: Добавлены все ожидаемые ответы (responses) для документации.
@app.post(
    "/translate",
    responses={
        200: {"description": "Успешный перевод"},
        400: {"description": "Ошибка клиента (неверный тип, невалидный ввод)"},
        500: {"description": "Ошибка сервера (сбой LLM API, невалидный JSON от LLM)"},
    },
)
async def translate_endpoint(request: TranslationRequest) -> TranslationResponse:
    # ❌ БЫЛО: Отсутствие ограничения конкурентности.
    # ✅ ИСПРАВЛЕНО: Использование Semaphore для контроля нагрузки.
    async with CONCURRENCY_SEMAPHORE:
        translated = None

        try:
            if isinstance(request.text, str):
                translated = await translate_string(
                    request.text, request.source_lang, request.target_lang
                )

            elif isinstance(request.text, (list, dict)):
                translated = await translate_structured_batch(
                    request.text, request.source_lang, request.target_lang
                )

            else:
                # ❌ БЫЛО: Некорректная обработка или raise общего Exception.
                # ✅ ИСПРАВЛЕНО: Явно рейзим 400 Bad Request для ошибок клиента.
                raise HTTPException(
                    status_code=status.HTTP_400_BAD_REQUEST,
                    detail="Неподдерживаемый тип данных. Требуется строка, список или словарь.",
                )

        except HTTPException:
            # ✅ КОРРЕКТНО: Если была поднята HTTPException (например, 400), передаем ее дальше.
            raise

        except Exception as e:
            # ❌ БЫЛО: Общая обработка без специфики.
            # ✅ ИСПРАВЛЕНО: Ловим все исключения, поднятые LLM-функциями, и маппим их в 500.
            logger.error(f"Translation processing error: {e}")
            raise HTTPException(
                status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
                detail=f"Внутренняя ошибка сервиса: {e}",
            )

        # ✅ КОРРЕКТНО: В случае успеха возвращаем 200 OK.
        return TranslationResponse(
            original=request.text, translated=translated, status="success"
        )


@app.get("/status")
def get_status():
    # ✅ ИСПРАВЛЕНО: Возврат статуса конкурентности на основе Semaphore.
    current_active = MAX_CONCURRENT_REQUESTS - CONCURRENCY_SEMAPHORE._value

    return {
        "active_requests": current_active,
        "max_concurrent": MAX_CONCURRENT_REQUESTS,
        "status": "ok",
    }


if __name__ == "__main__":
    import uvicorn

    # ❌ БЫЛО: if name == "main": (неверный синтаксис).
    # ✅ ИСПРАВЛЕНО: Стандартный запуск.
    uvicorn.run(app, host="0.0.0.0", port=8000)